# 🗞️ AI Morning News Brief Notebook
This is a standalone notebook compiling the entire `AI Morning News Brief` project so you can run the pipeline directly here.

### 1. Install Dependencies & Setup Environment

In [ ]:
!pip install groq requests python-dotenv rich nbformat

In [ ]:
import os
from dotenv import load_dotenv

# Load existing .env if present
load_dotenv()

# Set API Keys here if not using .env
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
    
if not os.getenv("NEWSAPI_KEY"):
    os.environ["NEWSAPI_KEY"] = ""  # Optional


### `config.py`

In [ ]:
# Source: config.py

"""
Configuration — Loads API keys and project settings.
"""

import os
from dotenv import load_dotenv

load_dotenv()

# --- API Keys ---
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY", "")

# --- LLM Settings ---
GROQ_MODEL = "openai/gpt-oss-120b"
TEMPERATURE = 0.7
MAX_TOKENS = 1024

# --- News Settings ---
MAX_ARTICLES = 2
ARTICLE_MAX_CHARS = 3000


def validate_config():
    """Ensure required keys are set before running."""
    if not GROQ_API_KEY:
        raise ValueError(
            "❌ GROQ_API_KEY not found!\n"
            "   Add it to your .env file.\n"
            "   Get a free key at: https://console.groq.com/keys"
        )

### `utils/prompts.py`

In [ ]:
# Source: utils/prompts.py

"""
Prompt Manager — All prompt templates in one place.
Easy to edit and iterate without touching any logic.
"""


def summarize_all_prompt(articles_text: str) -> str:
    """Prompt to summarize multiple articles at once and return JSON."""
    return f"""You are a news analyst. Below is a list of news articles. 
Summarize EACH article in exactly 2-3 concise sentences. Focus on key facts.

Return your response ONLY as a valid JSON array of objects, with no markdown formatting.
Each object must have exactly these keys: "title", "source", "summary".

Articles:
{articles_text}"""


def morning_brief_prompt(summaries_text: str, topic: str) -> str:
    """Prompt to combine summaries into a clean morning brief."""
    return f"""You are a news editor creating a morning brief about "{topic}".

Below are summaries of today's top stories:

{summaries_text}

Write a short, engaging morning brief (4-6 sentences) that ties these stories together.
Write it as a cohesive paragraph, not as a list. Make it sound like a professional news anchor.

Morning Brief:"""


def why_it_matters_prompt(brief_text: str, topic: str) -> str:
    """Prompt to generate a 'Why It Matters' section."""
    return f"""Based on this morning news brief about "{topic}":

{brief_text}

Write a "Why It Matters" section in exactly 2-3 sentences.
Explain why an average reader should care about these developments.
Focus on real-world impact — jobs, daily life, future implications.

Why It Matters:"""

### `data/sample_articles.py`

In [ ]:
# Source: data/sample_articles.py

"""
Sample news articles for development and testing (Phase A).
Used when no live API is available.
"""

SAMPLE_ARTICLES = [
    {
        "title": "AI Transforms Healthcare Diagnostics",
        "source": "TechHealth Today",
        "content": (
            "Artificial intelligence is rapidly transforming healthcare by improving "
            "diagnostic accuracy and enabling early detection of diseases. Hospitals are "
            "increasingly adopting AI-powered tools to assist doctors in decision-making. "
            "Recent studies show that AI systems can detect certain cancers with 95% accuracy, "
            "outperforming traditional screening methods. However, concerns remain about data "
            "privacy, algorithmic bias, and over-reliance on automated systems."
        ),
    },
    {
        "title": "Governments Push for AI Regulation",
        "source": "Global Policy Review",
        "content": (
            "Governments worldwide are introducing new regulations to control the development "
            "of artificial intelligence. The European Union's AI Act has become the global "
            "benchmark for responsible AI governance. Critics argue that excessive regulation "
            "could slow innovation and reduce competitiveness. Companies are pushing for "
            "balanced policies that protect consumers without stifling progress."
        ),
    },
    {
        "title": "Tech Giants Invest Billions in AI",
        "source": "Market Pulse",
        "content": (
            "Tech companies are investing billions into AI research, leading to rapid "
            "advancements in machine learning and automation. Google, Microsoft, and OpenAI "
            "are competing fiercely for talent and breakthroughs. These innovations are "
            "expected to boost productivity across industries. At the same time, fears about "
            "job displacement are growing, with analysts suggesting that reskilling workers "
            "will be essential to adapt to the changing job market."
        ),
    },
    {
        "title": "AI in Education: Promise and Concerns",
        "source": "EduTech Weekly",
        "content": (
            "Schools are integrating AI tools into classrooms, offering personalized learning "
            "experiences. AI tutors can adapt to individual paces and identify knowledge gaps. "
            "However, educators worry about academic integrity as AI-generated content makes "
            "plagiarism harder to detect. Finding the right balance between AI benefits and "
            "educational standards remains a challenge."
        ),
    },
    {
        "title": "The Environmental Cost of AI",
        "source": "Green Future Report",
        "content": (
            "The expansion of AI has brought attention to its environmental footprint. "
            "Training large AI models consumes vast amounts of electricity and water. "
            "Estimates suggest training a single large language model can emit as much carbon "
            "as five cars over their lifetimes. Researchers are developing energy-efficient "
            "architectures, and some companies have pledged carbon-neutral AI by 2030."
        ),
    },
     {
        "title": "Global Temperatures Hit Record High in 2026",
        "source": "Climate Watch News",
        "content": (
            "Scientists have reported that global average temperatures have reached a new "
            "record high in 2026, raising concerns about the accelerating impacts of climate "
            "change. Heatwaves have become more frequent across Asia, Europe, and North "
            "America, causing health risks and increasing energy demand. Experts warn that "
            "without stronger emission cuts, the world may exceed the 1.5°C warming threshold "
            "within the next decade. Governments are under pressure to strengthen climate "
            "policies and invest more heavily in renewable energy solutions."
        ),
    },
    {
        "title": "Rising Sea Levels Threaten Coastal Cities",
        "source": "Oceanic Times",
        "content": (
            "New studies show that sea levels are rising faster than previously expected, "
            "threatening major coastal cities worldwide. Regions such as Bangladesh, the "
            "Maldives, and parts of Florida are facing increased flooding and erosion. "
            "Scientists highlight that melting polar ice caps and thermal expansion of ocean "
            "water are key contributors. Many countries are now planning large-scale coastal "
            "defense projects, but experts warn that adaptation alone will not be enough "
            "without reducing greenhouse gas emissions."
        ),
    },
]

### `services/llm_service.py`

In [ ]:
# Source: services/llm_service.py

"""
LLM Service — Central wrapper for all AI calls via Groq.
Every LLM interaction in the project goes through call_llm().
"""

import time
from groq import Groq

# Initialize the Groq client
client = Groq(api_key=GROQ_API_KEY)


def call_llm(prompt: str, temperature: float = None, max_retries: int = 3) -> str:
    """
    Send a prompt to Groq API and return the response text.

    Args:
        prompt: The prompt string to send.
        temperature: Override default creativity level.
        max_retries: Retry attempts on failure.

    Returns:
        LLM response as a plain string.
    """
    temp = temperature if temperature is not None else TEMPERATURE

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            print("    🤖 Calling Groq API...")
            time.sleep(1) # Delay to prevent rate limits
            
            response = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temp,
            )
            print("    ✅ Response received")
            return response.choices[0].message.content.strip()
            
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                # 1st retry -> 2 seconds, 2nd retry -> 4 seconds
                wait = 2 * attempt 
                print(f"    ⚠ Groq API failed: {e}")
                print(f"    🔄 Retrying in {wait}s...")
                time.sleep(wait)

    raise RuntimeError(f"Groq LLM failed after {max_retries} attempts: {last_error}")

### `services/news_fetcher.py`

In [ ]:
# Source: services/news_fetcher.py

"""
News Fetcher — Retrieves news articles.
Phase A: Returns hardcoded sample articles.
Phase B: Fetches live articles from NewsAPI.
"""

import requests


def fetch_articles(topic: str, use_api: bool = False) -> list[dict]:
    if use_api and NEWSAPI_KEY:
        return _fetch_from_api(topic)
    return _fetch_samples()


def _fetch_samples() -> list[dict]:
    """Phase A: Return hardcoded sample articles."""
    print("  📂 Using sample articles (Phase A)")
    return SAMPLE_ARTICLES[:MAX_ARTICLES]


def _fetch_from_api(topic: str) -> list[dict]:
    """Phase B: Fetch articles from NewsAPI."""
    print(f"  🌐 Fetching from NewsAPI: '{topic}'")

    url = "https://newsapi.org/v2/everything"
    params = {
        "q": topic,
        "language": "en",
        "sortBy": "relevancy",
        "pageSize": MAX_ARTICLES,
        "apiKey": NEWSAPI_KEY,
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        articles = []
        for item in data.get("articles", [])[:MAX_ARTICLES]:
            content = item.get("content") or item.get("description") or ""
            articles.append({
                "title": item.get("title", "Untitled"),
                "source": item.get("source", {}).get("name", "Unknown"),
                "content": content[:ARTICLE_MAX_CHARS],
            })

        print(f"  ✅ Got {len(articles)} articles")
        return articles

    except requests.RequestException as e:
        print(f"  ❌ API failed: {e}")
        print("  📂 Falling back to sample articles")
        return _fetch_samples()

### `agents/summarizer.py`

In [ ]:
# Source: agents/summarizer.py

"""
Summarizer Agent — Summarizes individual news articles.
Takes one article → returns a 2-3 line summary via the LLM.
"""

import json

def summarize_all(articles: list[dict]) -> list[dict]:
    """
    Summarize a list of articles using a single LLM API call.

    Returns:
        List of dicts with title, source, summary.
    """
    print(f"  📝 Summarizing {len(articles)} articles in a single batch...")
    
    # Combine all articles into a single prompt string
    articles_text = ""
    for i, article in enumerate(articles, 1):
        title = article.get("title", "Untitled")
        source = article.get("source", "Unknown")
        content = article.get("content", "")
        # Truncation limits were already handled in news_fetcher
        articles_text += f"---\nArticle {i}:\nTitle: {title}\nSource: {source}\nContent: {content}\n"
    
    prompt = summarize_all_prompt(articles_text)
    
    # Make 1 LLM call instead of N calls
    response_text = call_llm(prompt, temperature=0.3)
    
    # Clean up JSON (Gemini sometimes returns markdown block '```json...```')
    clean_json = response_text.replace("```json", "").replace("```", "").strip()
    
    try:
        results = json.loads(clean_json)
        return results
    except json.JSONDecodeError as e:
        print(f"  ❌ Failed to parse JSON from LLM: {e}\nRaw output was: {response_text}")
        return []

### `agents/brief_generator.py`

In [ ]:
# Source: agents/brief_generator.py

"""
Brief Generator — Combines summaries into a morning brief + "Why It Matters".
Two simple functions, two LLM calls.
"""



def generate_brief(summaries: list[dict], topic: str) -> str:
    """
    Combine article summaries into a cohesive morning brief.

    Args:
        summaries: List of dicts with 'summary' key.
        topic: The news topic.

    Returns:
        Morning brief as a string.
    """
    print("  🔗 Creating morning brief...")

    # Format summaries for the prompt
    summaries_text = "\n\n".join(
        f"• {s['title']}: {s['summary']}" for s in summaries
    )

    prompt = morning_brief_prompt(summaries_text, topic)
    brief = call_llm(prompt, temperature=0.5)
    return brief


def generate_why_it_matters(brief: str, topic: str) -> str:
    """
    Generate a 'Why It Matters' section based on the brief.

    Args:
        brief: The morning brief text.
        topic: The news topic.

    Returns:
        "Why It Matters" text as a string.
    """
    print("  🧠 Generating 'Why It Matters'...")

    prompt = why_it_matters_prompt(brief, topic)
    why = call_llm(prompt, temperature=0.6)
    return why

### `utils/formatter.py`

In [ ]:
# Source: utils/formatter.py

"""
Output Formatter — Renders the final morning brief in a clean format.
Uses Rich for colorful, structured terminal output.
"""

from rich.console import Console
from rich.panel import Panel
from rich import box

console = Console()


def format_output(
    topic: str,
    summaries: list[dict],
    brief: str,
    why_it_matters: str,
) -> None:
    """
    Render the full morning brief to terminal.

    Args:
        topic: The news topic.
        summaries: List of article summary dicts.
        brief: The combined morning brief.
        why_it_matters: The "Why It Matters" text.
    """
    console.print()

    # ── Header ──
    console.print(
        Panel(
            f"[bold white]🗞️  AI Morning News Brief[/bold white]\n"
            f"[dim]Topic: {topic}[/dim]",
            border_style="cyan",
            box=box.DOUBLE,
            padding=(1, 2),
        )
    )
    console.print()

    # ── 🔥 Top Stories ──
    console.print("[bold yellow]🔥 Top Stories[/bold yellow]")
    console.print("─" * 50, style="dim")
    for i, s in enumerate(summaries, 1):
        console.print(f"\n  [bold cyan]{i}. {s['title']}[/bold cyan]")
        console.print(f"     [dim]— {s['source']}[/dim]")
    console.print()

    # ── ⚡ Quick Summary ──
    console.print(
        Panel(
            "\n".join(f"• {s['summary']}" for s in summaries),
            title="[bold magenta]⚡ Quick Summary[/bold magenta]",
            border_style="magenta",
            box=box.ROUNDED,
            padding=(1, 2),
        )
    )
    console.print()

    # ── 📰 Morning Brief ──
    console.print(
        Panel(
            brief,
            title="[bold green]📰 Morning Brief[/bold green]",
            border_style="green",
            box=box.ROUNDED,
            padding=(1, 2),
        )
    )
    console.print()

    # ── 🧠 Why It Matters ──
    console.print(
        Panel(
            why_it_matters,
            title="[bold red]🧠 Why It Matters[/bold red]",
            border_style="bright_red",
            box=box.HEAVY,
            padding=(1, 2),
        )
    )
    console.print()
    console.rule("[dim]End of Brief[/dim]", style="cyan")
    console.print()

### `main.py`

In [ ]:
# Source: main.py

"""
AI Morning News Brief — Main Pipeline
Orchestrates the full workflow: fetch → summarize → brief → format.

Usage:
    python main.py                          # Interactive (asks for topic)
    python main.py "AI in healthcare"       # Direct with topic
    python main.py --api "climate change"   # Use live NewsAPI
"""

import sys

# Force UTF-8 output for emojis on Windows
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')



def run(topic: str, use_api: bool = False) -> None:
    """Run the full morning brief pipeline."""

    print(f"\n{'='*50}")
    print(f"  🗞️  AI MORNING NEWS BRIEF")
    print(f"  Topic: {topic}")
    print(f"  Mode: {'Live API' if use_api else 'Sample Articles'}")
    print(f"{'='*50}\n")

    # Step 1: Fetch articles
    print("📰 Step 1: Fetching articles...")
    articles = fetch_articles(topic, use_api=use_api)
    if not articles:
        print("❌ No articles found. Exiting.")
        return
    print(f"   → {len(articles)} articles found\n")

    # Step 2: Summarize each article
    print("📝 Step 2: Summarizing articles...")
    summaries = summarize_all(articles)
    print(f"   → {len(summaries)} summaries ready\n")

    # Step 3: Generate morning brief
    print("📰 Step 3: Creating morning brief...")
    brief = generate_brief(summaries, topic)
    print("   → Brief ready\n")

    # Step 4: Generate "Why It Matters"
    print("🧠 Step 4: Generating 'Why It Matters'...")
    why = generate_why_it_matters(brief, topic)
    print("   → Done\n")

    # Step 5: Format and display
    print("🎨 Step 5: Rendering output...\n")
    format_output(
        topic=topic,
        summaries=summaries,
        brief=brief,
        why_it_matters=why,
    )

    print("✅ Pipeline complete!\n")


def main():
    """Entry point — handles CLI arguments."""
    validate_config()

    use_api = "--api" in sys.argv
    args = [a for a in sys.argv[1:] if a != "--api"]

    if args:
        topic = " ".join(args)
    else:
        topic = input("🎯 Enter a news topic: ").strip()
        if not topic:
            print("❌ No topic provided. Exiting.")
            return

    run(topic, use_api=use_api)

### 🎉 Run the Pipeline
You can now run the pipeline by executing the `run()` function below!

In [ ]:
run("AI in Healthcare", use_api=False)